[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/junlinghu/book-agents/blob/main/tutorial/13-observability.ipynb)

## Colab setup

On Google Colab, use the setup in `tutorial/common/colab.py`. A fresh Colab runtime does not have this repository, so clone the repo into the session first. Then, before the lesson cells, run `setup_colab` from that module (import it from `tutorial.common.colab` and call it, or open the file and run `setup_colab` from there). The helper installs the tutorial packages, checks the repo out under `/content/book-agents` when it is still missing, and copies a Colab secret named `OPENAI_API_KEY`. Local Jupyter and VS Code can skip it when you already have the repo.

An API key is required. Set `OPENAI_API_KEY` in Colab secrets (the key icon, secret name `OPENAI_API_KEY`) before you run `setup_colab`. Locally, put the same name in the repository-root `.env`. A missing key stops the notebook. The key is not printed.

Edits you make in Colab stay in that session. They do not push to GitHub.

# 13. Observability

When a restock goes wrong, you need the trace, not a guess about which tool ran. This notebook records spans inside `run_agent`.

Each span has a trace id, a status, and three identities: the user (`counter-lead`, shop staff), the agent (`shop-concierge`), and the tool. The low-stock question from tutorial 3 runs again, with staff asking whether oat milk may ship to a guest, so you can see the same work with a trace attached. The shelf tools and the loop are imported above. Nothing secret belongs in the JSON.

Shared helpers this lesson needs are imported, not copied from earlier notebooks. You can run this file by itself.

## Setup

This notebook calls the OpenAI Chat Completions API. `OPENAI_API_KEY` is required. Locally the key is loaded from the repository-root `.env` by `tutorial/common/client.py`. On Colab, `setup_colab` in `tutorial/common/colab.py` reads a secret of the same name. The value is not printed. A missing key stops the run.

Companion notes: `13-observability.md`.

In [ ]:
import sys
from pathlib import Path


def find_root():
    """Repo root, whether the kernel started here or in tutorial."""
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        marker = candidate / "tutorial" / "common" / "client.py"
        if marker.is_file():
            return candidate
    raise RuntimeError(
        "Open this notebook inside the book-agents repository. "
        "The setup cell looks for tutorial/common/client.py."
    )


ROOT = find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from tutorial.common.client import describe_runtime
from tutorial.runtime import chat, require_key

LESSON = '13-observability'
LESSON_NUMBER = 13
require_key()
print("tutorial:", LESSON)
print(describe_runtime())


## Shared pieces

These imports are the earlier pieces this lesson uses. They come from `tutorial.common` and `tutorial.runtime`. You do not need to run the previous notebook first.


In [ ]:
import json

from tutorial.common.harness import check, system_text
from tutorial.common.loop import run_agent
from tutorial.common.shelf import reset_db


## Tutorial 13 — trace spans

Each span names the user, the agent, and the tool. Secrets are not fields.

In [ ]:
from tutorial.common.harness import set_hook


def span_status(result):
    if result.startswith("ERROR") or result.startswith("DENIED") or result.startswith("CONFIRM_REQUIRED"):
        if result.startswith("CONFIRM_REQUIRED"):
            return "confirm_required"
        if result.startswith("DENIED") or "DENIED" in result[:80]:
            return "denied"
        return "error"
    return "ok"


def make_span(trace_id, step, name, result):
    """One tool span. Identity is explicit. Secret-shaped values are not copied in."""
    return {
        "trace_id": trace_id,
        "span_id": trace_id + "-" + str(step) + "-" + name,
        "service": "hearth-lane-concierge",
        "name": "tool." + name,
        "status": span_status(result),
        "step": step,
        "user": {"kind": "user", "id": "counter-lead"},
        "agent": {"kind": "agent", "id": "shop-concierge"},
        "actor": {"kind": "tool", "id": name},
    }


def root_span(trace_id):
    return {
        "trace_id": trace_id,
        "span_id": trace_id + "-turn",
        "service": "hearth-lane-concierge",
        "name": "concierge.turn",
        "status": "ok",
        "user": {"kind": "user", "id": "counter-lead"},
        "agent": {"kind": "agent", "id": "shop-concierge"},
        "actor": {"kind": "agent", "id": "shop-concierge"},
    }


set_hook("make_span", make_span)
set_hook("root_span", root_span)


## Run

`system_text()` mentions only the tools imported above. Run this cell after the ones above. Each model turn calls the Chat Completions API. A check prints a warning when the wording differs from the expected trace, and the notebook continues.

In [ ]:
reset_db()
result = run_agent(
    "Which products are at or below their reorder point, and may we ship oat milk to a guest?",
    system_text(),
    trace_id="trace-low-stock",
)
print("ANSWER:", result["text"])
print("SPANS:")
print(json.dumps(result["spans"], indent=2, sort_keys=True))
blob = json.dumps(result["spans"])
check(result["stopped"] == "final", "traced question finished")
check(len(result["spans"]) >= 2, "the turn span and at least one tool span were recorded")
check(all(span["user"]["id"] == "counter-lead" for span in result["spans"]), "every span names the user")
check(all(span["agent"]["id"] == "shop-concierge" for span in result["spans"]), "every span names the agent")
check(any(span["actor"]["kind"] == "tool" for span in result["spans"]), "a tool has its own actor")
check("sk-" not in blob and "canary" not in blob.lower(), "the trace does not carry a secret")
check("does not ship" in result["text"].lower(), "the answer is still grounded")
